# Preventing Data Leakage in scikit-learn
## A hands-on demonstration of `fit`, `transform`, and `fit_transform`

This notebook illustrates the difference between:
 - `fit()` — learning parameters from the training data,
 - `transform()` — applying the learned transformation,
 - `fit_transform()` — a shortcut that combines both.
 
 We'll also emphasize the importance of **no data leakage**:  
 we always fit transformers on training data only.

In [7]:
# Load some data and split into training and test sets
# Import libraries
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Load a scikit-learn dataset and split the dataset
X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=0
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("Data before scaling (first 3 rows of training set):\n", X_train[:3])


Training shape: (112, 4)
Test shape: (38, 4)
Data before scaling (first 3 rows of training set):
 [[5.  2.  3.5 1. ]
 [4.9 3.  1.4 0.2]
 [4.4 2.9 1.4 0.2]]


## Using the standard scikit-learn libraries' transformers.

_Note: In scikit-learn, the term “transformer” means:  an object that learns a transformation from data and can apply it to new data (via fit + transform)._

Here are the main 10 scikit-learn transformers most commonly used in real ML workflows and teaching:

- StandardScaler – standardizes features (zero mean, unit variance)
- MinMaxScaler – rescales features to a fixed range (default [0, 1])
- RobustScaler – scales using median and IQR (robust to outliers)
- OneHotEncoder – encodes categorical variables as one-hot vectors
- OrdinalEncoder – encodes categories as integer values
- SimpleImputer – fills in missing values (mean, median, or constant)
- PCA (Principal Component Analysis) – reduces dimensionality
- PolynomialFeatures – generates polynomial and interaction features
- TfidfVectorizer – converts text to TF–IDF feature vectors
- FunctionTransformer – wraps any custom Python function as a transformer

### Fit and transform step-by-step (no data leakage)

In [8]:

# We create a StandardScaler to normalize features (zero mean, unit variance).
scaler = StandardScaler()

# Fit on the training data only → learns mean and std from training set
scaler.fit(X_train)

# Transform both training and test data using the same scaler
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nMeans learned from training data:\n", scaler.mean_)
print("\nTransformed training data (first 3 rows):\n", X_train_scaled[:3])
print("\nTransformed test data (first 3 rows):\n", X_test_scaled[:3])


Means learned from training data:
 [5.86607143 3.03928571 3.77053571 1.20982143]

Transformed training data (first 3 rows):
 [[-1.01099825 -2.30005392 -0.15148942 -0.27585067]
 [-1.12773206 -0.08694362 -1.32740725 -1.32760471]
 [-1.71140115 -0.30825465 -1.32740725 -1.32760471]]

Transformed test data (first 3 rows):
 [[-0.89426443  0.7983005  -1.27141116 -1.32760471]
 [-1.24446588 -0.08694362 -1.32740725 -1.45907396]
 [-0.66079679  1.46223359 -1.27141116 -1.32760471]]


### Using `fit_transform` (shortcut)
`fit_transform()` is equivalent to calling `fit()` followed by `transform()`.
It is often used directly on the training set, never on test data to avoid data leakage.


In [9]:

# `fit_transform()` is equivalent to calling `fit()` followed by `transform()`.     
scaler2 = StandardScaler()
X_train_scaled_alt = scaler2.fit_transform(X_train)

# Confirm both give the same result
print("\nSame result as separate fit+transform:",
      np.allclose(X_train_scaled, X_train_scaled_alt))  # np.allclose for floating point comparison




Same result as separate fit+transform: True


## Combining multiple steps with Pipeline to avoid data leakage

In machine learning workflows, data preprocessing often involves multiple sequential steps — for example:
1. handling missing values,
2. generating additional features,
3. scaling or normalizing,
4. and finally training a model.

To keep this process clean, reproducible, and **free of data leakage**, scikit-learn provides the **`Pipeline`** object.

A `Pipeline` chains multiple transformers and a final estimator into one composite model.  
When you call:
- `fit()`: each step up to the last one calls its own `fit()` (and `transform()` if available) on the training data.
- `transform()` or `predict()`: the data flows through all transformations in the same order using the learned parameters.

This guarantees:
- consistent preprocessing during training and testing,
- no accidental leakage of information from the test set,
- and easy parameter tuning with `GridSearchCV` or `cross_val_score`.

Example structure:
**imputation → feature generation → scaling → model**

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.datasets import load_iris  # replace with your data

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scale", StandardScaler()),
    ("model", LinearRegression())
])

pipe.fit(X_train, y_train)              # fits all steps on training set only
y_pred = pipe.predict(X_test)           # transforms X_test with learned steps

### Custom transformer example
Let's define a simple transformer that replaces missing values with column means.
This shows how `fit` and `transform` are implemented under the hood.

In [11]:

from sklearn.base import BaseEstimator, TransformerMixin

class MeanImputer(BaseEstimator, TransformerMixin):
    """Custom transformer that fills missing values with column means."""
    def fit(self, X, y=None):
        """Learn the column means from the data."""
        self.col_means_ = np.nanmean(X, axis=0)
        return self
    
    def transform(self, X):
        """Replace missing values with the learned column means."""
        X = np.array(X, copy=True)
        inds = np.where(np.isnan(X))
        X[inds] = np.take(self.col_means_, inds[1])
        return X



In [12]:
# Example usage

# Generate some data with missing values
X_missing = np.array([[1.0, np.nan], [2.0, 5.0], [np.nan, 7.0]])


imputer = MeanImputer()    # Create the transformer
imputer.fit(X_missing)     # Learn the column means
print("\nColumn means learned by imputer:", imputer.col_means_)

X_filled = imputer.transform([[np.nan, 9.0], [3.0, np.nan]])   # Transform new data
print("\nTransformed data:\n", X_filled)


Column means learned by imputer: [1.5 6. ]

Transformed data:
 [[1.5 9. ]
 [3.  6. ]]


## Summary
 - Always call `fit` on training data only (to prevent data leakage).
 - Then call `transform` on both training and test data.
 - Use `fit_transform` as a shorthand for the training set.